In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from tqdm.notebook import tqdm
import scipy
import numba
import molsim

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 3: Coupled harmonic oscillators
A single harmonic oscillator has energy levels 0, ϵ, 2ϵ, · · · , ∞ (ϵ > 0). All harmonic oscillators in the
system can exchange energy.


<figure id="fig:ljphasediagram">
  <img src="./oscillator.png" alt="oscillator" style="width:60%;">
  <figcaption>A system of N harmonic oscillators with a total energy U.</figcaption>
</figure>


<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

## Question 1
Invent a computational scheme for the update of the system at constant total energy ($U$). Compare your scheme with the scheme that is incorporated into the given computer code.


In [ ]:
# @numba.jit
def NVE(numberOfOscillators, numberOfCycles, totalEnergy):
    """Calculate the distribution of energy in a system of oscillators for a fixed total energy

    Parameters:
        -numberOfOscillators (int): Number of oscillators in the system.
        -numberOfCycles (int): Number of cycles to run the simulation.
        -totalEnergy (int): Total energy to be distributed among the oscillators.

    Returns:
        np.ndarray: Array containing the energy distribution of the oscillators over the cycles.
    """
    initSteps = int(np.floor(0.5 * numberOfCycles))
    sum = 0.0
    count = 0.0
    oscillator = np.zeros(numberOfOscillators, dtype=np.int32)
    distribution = np.zeros(totalEnergy + 1, dtype=np.float64)

    # Distribute the total energy among oscillators
    for energy in range(totalEnergy):
        oscillator[energy % numberOfOscillators] += 1

    for cycle in range(1, numberOfCycles):
        for _ in range(numberOfOscillators):
            # Select two oscillators
            oscA, oscB = np.random.choice(numberOfOscillators, 2, replace=False)

            # Choose a random exchange direction
            flip = -1 if np.random.rand() < 0.5 else 1

            # If energy will not go negative, accept exchange
            if min(oscillator[oscA] + flip, oscillator[oscB] - flip) >= 0.0:
                oscillator[oscA] += flip
                oscillator[oscB] -= flip

            # Update the distribution after half the cycles
            if cycle > initSteps:
                distribution[oscillator[0]] += 1.0
                sum += oscillator[0]
                count += 1.0

    return sum / count, distribution / count

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

## Question 2
Investigate the plot of the energy distribution of the first oscillator as a function of the number of oscillators for a constant value of $U/N$ . Which distribution is recovered when $N$ becomes  large ? What is the function of the other $N − 1$ harmonic oscillators? Explain.

In [ ]:
# start refactor
nve_cycles = None
energyPerOscillator = None
oscillatorNumbers = None
# end refactor

fig, ax = plt.subplots()

for numberOfOscillators in oscillatorNumbers:
    energy, distribution = NVE(numberOfOscillators, nve_cycles, numberOfOscillators * energyPerOscillator)
    ax.plot(distribution, label=f"N={numberOfOscillators}")

ax.set_xlabel("Energy level", fontsize=20)
ax.set_ylabel("Energy / a.u", fontsize=20)
ax.set_xlim([0, 10])
ax.set_ylim([0, 0.5])
ax.legend()